# Pocket Screener Model Training

Notebook offline untuk feature engineering indikator teknikal, training model scikit-learn, evaluasi accuracy/precision/recall, dan export model ke `public/model.onnx`.

In [ ]:
from pathlib import Path
import json
import numpy as np
import onnx
import onnxruntime as ort
import pandas as pd
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.pipeline import Pipeline

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV_PATH = ROOT / 'data' / 'historical_daily.csv'
MODEL_PATH = ROOT / 'public' / 'model.onnx'
METRICS_PATH = ROOT / 'public' / 'model_metrics.json'

FEATURE_COLUMNS = [
    'return_1d', 'return_5d', 'ma5_ratio', 'ma20_ratio',
    'volume_ratio_20', 'value_log', 'rsi_14', 'macd',
    'macd_signal', 'macd_histogram', 'bb_position',
    'atr_pct', 'vwap_ratio'
]

In [ ]:
def rsi(close: pd.Series, period: int = 14) -> pd.Series:
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / period, adjust=False).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    value = 100 - 100 / (1 + rs)
    return value.fillna(100).where(avg_gain != 0, 0)

def macd(close: pd.Series):
    ema_fast = close.ewm(span=12, adjust=False).mean()
    ema_slow = close.ewm(span=26, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal = macd_line.ewm(span=9, adjust=False).mean()
    return macd_line, signal, macd_line - signal

def add_features(group: pd.DataFrame) -> pd.DataFrame:
    group = group.sort_values('date').copy()
    close, high, low, volume = group['close'], group['high'], group['low'], group['volume']
    group['return_1d'] = close.pct_change()
    group['return_5d'] = close.pct_change(5)
    group['ma5'] = close.rolling(5).mean()
    group['ma20'] = close.rolling(20).mean()
    group['ma5_ratio'] = close / group['ma5'] - 1
    group['ma20_ratio'] = close / group['ma20'] - 1
    group['volume_ratio_20'] = volume / volume.rolling(20).mean()
    group['value_log'] = np.log1p(close * volume)
    group['rsi_14'] = rsi(close)
    group['macd'], group['macd_signal'], group['macd_histogram'] = macd(close)
    middle = close.rolling(20).mean()
    std = close.rolling(20).std(ddof=0)
    upper, lower = middle + 2 * std, middle - 2 * std
    group['bb_position'] = (close - lower) / (upper - lower)
    tr = pd.concat([(high - low), (high - close.shift(1)).abs(), (low - close.shift(1)).abs()], axis=1).max(axis=1)
    group['atr_pct'] = tr.ewm(alpha=1 / 14, adjust=False).mean() / close
    typical_price = (high + low + close) / 3
    group['vwap_ratio'] = close / ((typical_price * volume).cumsum() / volume.cumsum()) - 1
    return group

In [ ]:
raw = pd.read_csv(CSV_PATH, parse_dates=['date']).sort_values(['symbol', 'date'])
dataset = pd.concat([add_features(group) for _, group in raw.groupby('symbol', sort=False)], ignore_index=True)
dataset['next_close'] = dataset.groupby('symbol')['close'].shift(-1)
dataset['target'] = (dataset['next_close'] > dataset['close']).astype(int)
dataset = dataset.dropna(subset=FEATURE_COLUMNS + ['target', 'next_close']).reset_index(drop=True)
dataset[['symbol', 'date', 'target'] + FEATURE_COLUMNS].head()

In [ ]:
split_date = dataset['date'].quantile(0.8)
train = dataset[dataset['date'] <= split_date]
test = dataset[dataset['date'] > split_date]

X_train = train[FEATURE_COLUMNS].astype(np.float32)
y_train = train['target'].astype(int)
X_test = test[FEATURE_COLUMNS].astype(np.float32)
y_test = test['target'].astype(int)

model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=250,
        min_samples_leaf=8,
        class_weight='balanced_subsample',
        random_state=42,
        n_jobs=-1,
    )),
])
model.fit(X_train, y_train)
pred = model.predict(X_test)

metrics = {
    'model': 'RandomForestClassifier',
    'target': 'next_day_close_return_positive',
    'features': FEATURE_COLUMNS,
    'trainRows': int(len(train)),
    'testRows': int(len(test)),
    'splitDate': pd.Timestamp(split_date).date().isoformat(),
    'accuracy': float(accuracy_score(y_test, pred)),
    'precision': float(precision_score(y_test, pred, zero_division=0)),
    'recall': float(recall_score(y_test, pred, zero_division=0)),
}
metrics

In [ ]:
initial_type = [('float_input', FloatTensorType([None, len(FEATURE_COLUMNS)]))]
classifier = model.named_steps['classifier']
onnx_model = convert_sklearn(
    model,
    initial_types=initial_type,
    target_opset=15,
    options={id(classifier): {'zipmap': False}},
)
onnx.checker.check_model(onnx_model)
MODEL_PATH.write_bytes(onnx_model.SerializeToString())
METRICS_PATH.write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')

session = ort.InferenceSession(str(MODEL_PATH), providers=['CPUExecutionProvider'])
session.run(None, {session.get_inputs()[0].name: X_test.head(5).to_numpy(dtype=np.float32)})
MODEL_PATH, METRICS_PATH